# **Imports**

In [1]:
# add parent directory to path to import other utils
import sys, os
sys.path.append(os.path.abspath('..'))
sys.path.append(os.path.abspath('../data'))
from utils import set_seed
from trainvaltestsplits_utils import *

import os
import glob
import shutil
import pandas as pd
import geopandas as gpd
import numpy as np
import rasterio

# **Select Smoke Set**

In [16]:
##################################################
# Select Smoke Splits from Train/Val/Test Splits
##################################################

# set seed
seed = 111
set_seed(111)

# path to global labels
global_labels_path = r'../../data/earthscape_labels.csv'

# path to selected splits...
train_patch_path = r'../../models/splits/indomain_train.geojson'
val_patch_path = r'../../models/splits/indomain_val.geojson'
test_patch_path = r'../../models/splits/indomain_test.geojson'
cross_test_patch_path = r'../../models/splits/crossdomain_test.geojson'

# output directory path
output_dir = r'../../models/splits'


##### select and save smoke splits
train = make_smoke_set(global_labels_path, train_patch_path, split_size=32, threshold=0, seed=0)
train.drop(columns=train.columns[2:], inplace=True)
train.to_file(f"{output_dir}/smoke_train.geojson", driver='GeoJSON', index=False)

val = make_smoke_set(global_labels_path, val_patch_path, split_size=12, threshold=0, seed=0)
val.drop(columns=val.columns[2:], inplace=True)
val.to_file(f"{output_dir}/smoke_val.geojson", driver='GeoJSON', index=False)

test = make_smoke_set(global_labels_path, test_patch_path, split_size=16, threshold=0, seed=0)
test.drop(columns=test.columns[2:], inplace=True)
test.to_file(f"{output_dir}/smoke_test.geojson", driver='GeoJSON', index=False)

cross_test = make_smoke_set(global_labels_path, cross_test_patch_path, split_size=16, threshold=0, seed=0)
cross_test.drop(columns=cross_test.columns[2:], inplace=True)
cross_test.to_file(f"{output_dir}/smoke_cross_test.geojson", driver='GeoJSON', index=False)


# **Save Smoke Set Files**

In [17]:
##################################################
# Copy and save smoke set images to data folder.
##################################################

# paths to patches directories
dataset_dir = r'../../data'
area_dirs = [d for d in os.listdir(dataset_dir) if os.path.isdir(os.path.join(dataset_dir, d))]

# output directory
output_dir = r'../../data/smoke/patches_smoke'
if not os.path.isdir(output_dir):
    os.makedirs(output_dir)

# list of all smoke patch IDs
all_patch_ids = train['patch_id'].astype(str).to_list()\
                + val['patch_id'].to_list()\
                + test['patch_id'].astype(str).to_list()\
                + cross_test['patch_id'].astype(str).to_list()


##### get paths to data splits...
paths = []
for pid in all_patch_ids:
    for ad in area_dirs:
        candidate_dir = glob.glob(f"{dataset_dir}/{ad}/patches_*")[0]
        if os.path.isdir(candidate_dir):
            match_img = glob.glob(f"{candidate_dir}/{pid}_*.tif")
            match_csv = glob.glob(f"{candidate_dir}/{pid}_*.csv")
            if len(match_img) > 0:
                paths.extend(match_img)
                paths.extend(match_csv)
            

##### save data to smoke dataset directory...
for src in paths:
    basename = os.path.basename(src)
    dst = f"{output_dir}/{basename}"
    shutil.copy2(src, dst)
    

# **Smoke Set Local Files**

## *Areas, Labels, & Patch Locations*

In [18]:
#####################################################################################
# Save global files to mimic same structure as other local area dataset directories.
#####################################################################################

# global dataset file paths...
area_path = r'../../data/earthscape_areas.csv'
label_path = r'../../data/earthscape_labels.csv'
patches_path = r'../../data/earthscape_patches.geojson'

# smoke set paths...
smoke_train_path = r'../../models/splits/smoke_train.geojson'
smoke_val_path = r'../../models/splits/smoke_val.geojson'
smoke_test_path = r'../../models/splits/smoke_test.geojson'
smoke_cross_path = r'../../models/splits/smoke_cross_test.geojson'

# output directory
output_dir = r'../../data/smoke'


##### get smoke set patch IDs...
smoke_ids = []
for path in [smoke_train_path, smoke_val_path, smoke_test_path, smoke_cross_path]:
    gdf = gpd.read_file(path)
    ids = gdf['patch_id'].to_list()
    smoke_ids.extend(ids)


##### extract areas, labels, and patches files for smoke set...
area = pd.read_csv(area_path)
area = area.loc[area['patch_id'].isin(smoke_ids)]
area.to_csv(f"{output_dir}/smoke_256_50_areas.csv", index=False)

label = pd.read_csv(label_path)
label = label.loc[label['patch_id'].isin(smoke_ids)]
label.to_csv(f"{output_dir}/smoke_256_50_labels.csv", index=False)

patches = gpd.read_file(patches_path)
patches = patches.loc[patches['patch_id'].isin(smoke_ids)]
patches.to_file(f"{output_dir}/smoke_256_50_patches.geojson", driver='GeoJSON', index=False)

## *Image Stats*

In [39]:
#############################################################
# Get Mean & Standard Deviations of Images for Normalization
#############################################################

# data directory
data_dir = r'../../data/smoke'

# paths to images
image_paths = glob.glob(f"{data_dir}/patches_smoke/*.tif")
image_paths.sort(key=lambda x: x.lower())

# list of unique modality/channel basenames...
channel_names = glob.glob(f"{data_dir}/patches_smoke/*.tif")
channel_names = [os.path.basename(f) for f in channel_names]
channel_names = [f.split('_50_')[1] for f in channel_names]
channel_names = [f.split('_')[1:] for f in channel_names]
channel_names = ['_'.join(cn) for cn in channel_names]
channel_names = list(set(channel_names))
channel_names.sort()
channel_names[:40]

##### calculate mean and standard deviation for each image...
df = pd.DataFrame(columns=['channel', 'mean', 'sd'])

for idx, channel in enumerate(channel_names):
    
    image_paths = glob.glob(f"{data_dir}/patches_smoke/*{channel}")   # get paths to specific channels
    means = []
    sds = []

    for path in image_paths:
        with rasterio.open(path) as src:
            data = src.read(1, masked=True)          # read image data array
            valid_data = data[~data.mask].data       # get valid data only (masked array)
            means.append(np.mean(valid_data))        # calculate sample mean (each image is sample of population)
            sds.append(np.std(valid_data, ddof=1))   # calculate sample sd (each image is sample of population so df=1)

    # convert to numpy arrays...
    means = np.asarray(means)
    sds = np.asarray(sds)

    # overall mean using CLT
    overall_mean = np.mean(means)

    # sds**2 - within image variance | (means-overall_means)**2 - between image variance
    overall_sd = np.sqrt(np.mean(sds**2 + (means - overall_mean)**2))

    df.loc[idx, 'channel'] = channel
    df.loc[idx, 'mean'] = overall_mean
    df.loc[idx, 'sd'] = overall_sd


##### save image stats as .csv with patches
output_path = glob.glob(f"{data_dir}/*areas.csv")[0]
output_path = output_path.replace('areas', 'image_stats')
df.to_csv(output_path, index=False)
